# Udaplay_02 Solution Project
Demo of `GameAgent` using local vector retrieval with optional Tavily web fallback.

In [1]:
import os
import json
import logging
from pathlib import Path

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.basicConfig(level=logging.INFO)

from starter.lib.vector_db import VectorDB
from starter.lib.agent import GameAgent

PROJECT_DIR = Path(".").resolve()
GAMES_DIR = PROJECT_DIR / "games"

def load_games():
    games = []
    for path in sorted(GAMES_DIR.glob("*.json")):
        with path.open("r", encoding="utf-8") as f:
            game = json.load(f)
        if "Name" in game and "Description" in game:
            games.append(game)
    return games

db = VectorDB(collection_name="games_collection", persist_dir=str(PROJECT_DIR / "chromadb"))
games = load_games()
db.ingest(games)

agent = GameAgent(vector_db=db, tavily_api_key=os.getenv("TAVILY_API_KEY"))
len(games)


INFO:starter.lib.vector_db:No new documents to ingest.


15

In [2]:
queries = [
    "When was Super Mario World released?",
    "Which platform is Marvel's Spider-Man 2 available on?",
    "Who is the publisher of Grand Theft Auto: San Andreas?",
]
queries


['When was Super Mario World released?',
 "Which platform is Marvel's Spider-Man 2 available on?",
 'Who is the publisher of Grand Theft Auto: San Andreas?']

In [3]:
for q in queries:
    result = agent.answer_query(q, n_results=3)
    print(f"\nQuery: {q}")
    print(f"Source: {result["source"]}")
    for i, item in enumerate(result["results"], start=1):
        print(f"  {i}. {item}")


INFO:root:Using internal DB retrieval...


INFO:root:Using internal DB retrieval...


INFO:root:Using internal DB retrieval...



Query: When was Super Mario World released?
Source: internal_db
  1. {'doc': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'meta': {'Publisher': 'Nintendo', 'YearOfRelease': 1990, 'Genre': 'Platformer', 'Name': 'Super Mario World', 'Platform': 'Super Nintendo Entertainment System (SNES)'}}
  2. {'doc': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.', 'meta': {'Platform': 'Xbox One', 'YearOfRelease': 2014, 'Publisher': 'Mojang Studios', 'Name': 'Minecraft', 'Genre': 'Sandbox, Survival'}}
  3. {'doc': 'A collection of mini-games designed to showcase the capabilities of the Kinect motion sensor.', 'meta': {'YearOfRelease': 2010, 'Publisher': 'Microsoft Game Studios', 'Genre': 'Party', 'Name': 'Kinect Adventures!', 'Platform': 'Xbox 360'}}

Query: Which platform is Marvel's Spider-Man 2 available on?
Source: internal_db
  1. {'doc': 'A sandbox game that allows 